# Concatenate & Normalize ACLU Legislation Tracker (2021–2026)

In [23]:
import pandas as pd
import re

# --- File paths (update if needed) ---
files = {
    2021: '../aclu/legislation-tracker_2021.csv',
    2022: '../aclu/legislation-tracker_2022.csv',
    2023: '../aclu/legislation-tracker_2023.csv',
    2024: '../aclu/legislation-tracker_2024.csv',
    2025: '../aclu/legislation-tracker_2025.csv',
    2026: '../aclu/legislation-tracker_2026.csv',
}

In [24]:
# --- Column mapping ---
COLUMN_MAP = {
    "State":          "state",
    "Bill Name":      "bill_name",
    "Issues":         "issues",
    "Status":         "status",
    "Status Detail":  "status_detail",
    "Status Date":    "status_date",
    "In Court Link":  "in_court_link",
}

In [25]:
frames = []

for year, filename in files.items():
    df = pd.read_csv(filename)
    df.columns = df.columns.str.strip()
    df = df.rename(columns=COLUMN_MAP)

    # Add missing columns (2022 lacks status_detail)
    for col in COLUMN_MAP.values():
        if col not in df.columns:
            df[col] = pd.NA

    df["year"] = year

    # Strip whitespace from string columns
    str_cols = ["state", "bill_name", "issues", "status", "status_detail"]
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip().replace({"<NA>": pd.NA, "": pd.NA, "nan": pd.NA})

    frames.append(df)
    print(f"{year}: {df.shape[0]:,} rows")

# Concat — drop in_court_link right away
KEEP_COLS = ["year", "state", "bill_name", "issues", "status", "status_detail", "status_date"]
df_all = pd.concat(frames, ignore_index=True)[KEEP_COLS]
print(f"\nCombined (raw): {df_all.shape}")

2021: 511 rows
2022: 501 rows
2023: 511 rows
2024: 534 rows
2025: 617 rows
2026: 385 rows

Combined (raw): (3059, 7)


In [26]:
# ============================================================
# CLEAN: Remove non-state junk rows
# (NaN states, "Data is current as of..." footer rows)
# ============================================================
before = len(df_all)
df_all = df_all.dropna(subset=["state"])
df_all = df_all[~df_all["state"].str.startswith("Data", na=False)]
print(f"Removed {before - len(df_all)} junk rows → {len(df_all)} remaining")

Removed 544 junk rows → 2515 remaining


In [27]:
# ============================================================
# NORMALIZE: State names → 2-letter abbreviations
# ============================================================
STATE_TO_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
    # Territories
    "Puerto Rico": "PR", "District of Columbia": "DC",
    # Edge case in ACLU data
    "New England": "NE",  # likely Nebraska mislabeled
}
VALID_ABBRS = set(STATE_TO_ABBR.values())

def normalize_state(val):
    val = str(val).strip()
    if val.upper() in VALID_ABBRS:
        return val.upper()
    if val in STATE_TO_ABBR:
        return STATE_TO_ABBR[val]
    return pd.NA

df_all["state"] = df_all["state"].apply(normalize_state)

unmapped = df_all["state"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows with unmapped states — dropping")
    df_all = df_all.dropna(subset=["state"])

print(f"Unique states ({df_all['state'].nunique()}): {sorted(df_all['state'].unique())}")

Unique states (51): ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


In [28]:
# ============================================================
# ADD: label, source, parse dates
# ============================================================
df_all["label"] = "harmful"
df_all["source"] = "aclu"

df_all["status_date"] = pd.to_datetime(
    df_all["status_date"], format="mixed", dayfirst=False, errors="coerce"
)

df_all = df_all[
    ["year", "state", "bill_name", "issues", "label", "source", "status", "status_detail",
     "status_date"]
].reset_index(drop=True)

print(f"Final shape: {df_all.shape}")
df_all.head(10)

Final shape: (2515, 9)


,year,state,bill_name,issues,label,source,status,status_detail,status_date
0,2021,AL,SB 10,Anti-Trans bills | Prohibiting heathcare for t...,harmful,aclu,Introduced,Died,NaT
1,2021,AL,HB 1,Anti-Trans bills | Prohibiting heathcare for t...,harmful,aclu,Introduced,Indefinitely postponed,2021-05-06
2,2021,AL,SB 5,Anti-Trans bills | Prohibiting heathcare for t...,harmful,aclu,Introduced,Prefiled for 2022,2022-01-11
3,2021,AL,HB 391,Anti-Trans bills | Excluding transgender youth...,harmful,aclu,Introduced,Signed by Governor,NaT
4,2021,AR,SB 347,Anti-Trans bills | Prohibiting heathcare for t...,harmful,aclu,Introduced,Died in committee,2021-10-15
5,2021,AR,HB 1570,Anti-Trans bills | Prohibiting heathcare for t...,harmful,aclu,Introduced,Signed into law,2021-04-13
6,2021,AR,HB 1882,Anti-Trans bills | Single-Sex Facility Restric...,harmful,aclu,Introduced,Died,2021-10-15
7,2021,AR,HB 1905,Anti-Trans bills | Single-Sex Facility Restric...,harmful,aclu,Introduced,Referred to committee,2021-04-12
8,2021,AR,SJR 16,Anti-Trans bills | Excluding transgender youth...,harmful,aclu,Introduced,Died,2021-10-15
9,2021,AR,SB 354,Anti-Trans bills | Excluding transgender youth...,harmful,aclu,Introduced,Signed by Governor,2021-03-29


In [29]:
# --- Sanity checks ---
print("Null counts:")
print(df_all.isnull().sum())
print(f"\nUnique statuses: {sorted(df_all['status_detail'].dropna().unique())}")
print(f"\nYear distribution:\n{df_all['year'].value_counts().sort_index()}")

Null counts:
year               0
state              0
bill_name          0
issues             0
label              0
source             0
status            48
status_detail    328
status_date       52
dtype: int64

Unique statuses: ['Active', 'Amendment offered', 'Approved by Governor', 'Approved by the governor', 'Assembly amendment offered', 'Assigned to Senate subcommittee', 'Assigned to subcommittee', 'Became SB 228; Signed by Governor', 'Became act', "Became act without Governor's signature", 'Became act without Governor’s signature', 'Became law', "Became law without Governor's signature", 'Became law without signature', "Become law without Governor's signature", 'Bill renumbered HF 623', 'Bill renumbered SF 538', 'By resolution, reintroduced and retained in present status.', 'Carried over', 'Committee action deferred to', 'Committee hearing', 'Committee recommended ought not to pass by committee', 'Committee reported inexpedient to legislate, not retained for 2026 session', 'Co

In [30]:
# --- Export ---
df_all.to_csv("aclu_bills_2021_2026.csv", index=False)
print("Saved: aclu_bills_2021_2026.csv")

Saved: aclu_bills_2021_2026.csv


In [31]:
"""
ACLU Data Cleaning: Label Correction + Bill Number Normalization
================================================================
1. Reclassify pro-LGBTQ bills as 'supportive' (currently all labeled 'harmful')
2. Normalize bill_name to match LegiScan bill_number format per state
"""

# ============================================================
# LOAD
# ============================================================
aclu = pd.read_csv('aclu_bills_2021_2026.csv')
print(f"Total ACLU rows: {len(aclu)}")
print(f"Original label distribution:\n{aclu['label'].value_counts()}")


# ============================================================
# STEP 1: FIX LABELS — Reclassify pro-LGBTQ bills as 'supportive'
# ============================================================
# The `issues` column contains the signal. Bills with these patterns
# are PRO-LGBTQ (equality/protection bills), not harmful.

supportive_patterns = [
    'LGBTQ Equality Bills',
    'Nondiscrimination protections',
    'Allowing updated gender markers on ID',
    'Other good bills',
    'Protections in healthcare',
    # Decriminalizing laws that disproportionately target LGBTQ+ people
    'Sex work decriminalization bills',
    'Sex work discriminalization bills',   # Typo variant in the data
]


def classify_label(issues_str):
    """Determine if a bill is supportive or harmful based on issue categories."""
    if pd.isna(issues_str):
        return 'harmful'  # Default: most ACLU-tracked bills are anti-LGBTQ

    issues = str(issues_str)

    # Check if ANY supportive pattern appears
    for pattern in supportive_patterns:
        if pattern in issues:
            # But make sure it's not ALSO tagged with harmful categories
            # e.g., a bill could theoretically appear in both
            harmful_indicators = [
                'Anti-Trans', 'Excluding transgender', 'Prohibiting',
                'Restricting', 'Barriers', 'Bans', 'bans',
                'Religious exemptions', 'Weakening', 'Forced outing',
                'Curriculum censorship', 'Re-definition of sex',
                'Public accommodation bans', 'Prison healthcare restrictions',
                'Healthcare restrictions', 'Healthcare age restrictions',
                'Healthcare funding restrictions',
            ]
            has_harmful = any(h in issues for h in harmful_indicators)

            if not has_harmful:
                return 'supportive'

    return 'harmful'


aclu['label'] = aclu['issues'].apply(classify_label)

print(f"\nCorrected label distribution:\n{aclu['label'].value_counts()}")

# Show the supportive bills for verification
print(f"\n--- Supportive Bills Sample ---")
supportive = aclu[aclu['label'] == 'supportive']
print(supportive[['year', 'state', 'bill_name', 'issues']
                 ].to_string(index=False))


# ============================================================
# STEP 2: NORMALIZE BILL NUMBERS TO MATCH LEGISCAN FORMAT
# ============================================================
# LegiScan uses a consistent format per state. The ACLU data has
# inconsistent formats: "HB 1", "H.B. 1", "HB1", "H 1", etc.
#
# General LegiScan patterns observed from the data:
#   Most states: HB{num}, SB{num} (no space, no dots)
#   CA, NV, WI: AB{num}, SB{num}
#   CO: HB{num} with leading dash sometimes: HB21-1186 → need to check
#   CT: HB{num}, SB{num} (with leading zeros like HB05934)
#   IA: HF{num}, SF{num}, HSB{num}, SSB{num}
#   ID: H{num}, S{num} (just H/S prefix, no B)
#   MA: H{num}, S{num}, HD{num}
#   ME: HP{num}, LD{num}
#   MN: HF{num}, SF{num}
#   NC: H{num}, S{num}
#   NE: LB{num}
#   NJ: A{num}, S{num}
#   NY: A{num}, S{num}
#   PR: PS{num}, PC{num}
#   RI: H{num}, S{num}, HB{num}, SB{num}
#   SC: H{num}, S{num}
#   VT: H{num}, S{num}
#   WY: HB{num}, SF{num}
#   Joint resolutions: HJR{num}, SJR{num}, HCR{num}, SCR{num}, etc.

# State-specific prefix mappings: ACLU format → LegiScan format
# These map the various ACLU prefix formats to what LegiScan expects
STATE_PREFIX_MAP = {
    # States where LegiScan uses just H/S (no B)
    'ID': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
    'MA': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S', 'HD': 'HD'},
    'NC': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
    'NJ': {'AB': 'A', 'SB': 'S', 'A': 'A', 'S': 'S'},
    'NY': {'AB': 'A', 'SB': 'S', 'A': 'A', 'S': 'S'},
    'RI': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
    'SC': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
    'VT': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},

    # States with non-standard prefixes
    'CA': {'AB': 'AB', 'SB': 'SB'},
    'NV': {'AB': 'AB', 'SB': 'SB'},
    'WI': {'AB': 'AB', 'SB': 'SB'},
    'IA': {'HF': 'HF', 'SF': 'SF', 'HSB': 'HSB', 'SSB': 'SSB'},
    'MN': {'HF': 'HF', 'SF': 'SF'},
    'NE': {'LB': 'LB'},
    'ME': {'HP': 'HP', 'LD': 'LD'},
    'PR': {'PS': 'PS', 'PC': 'PC'},
    'WY': {'HB': 'HB', 'SF': 'SF'},

    # States where LegiScan omits the bill type (B is optional)
    'FL': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
    'KS': {'HB': 'H', 'SB': 'S', 'H': 'H', 'S': 'S'},
}


def normalize_bill_number(bill_name, state):
    """
    Normalize ACLU bill_name to match LegiScan bill_number format.
    
    Steps:
    1. Remove dots, extra spaces
    2. Extract prefix and number
    3. Apply state-specific prefix mapping
    4. Handle leading zeros for certain states
    5. Return clean format: PREFIX + NUMBER (no space)
    """
    if pd.isna(bill_name):
        return bill_name

    raw = str(bill_name).strip()

    # --- Clean up dots and normalize ---
    # "H.B. 1570" → "HB 1570", "S.J.R. 16" → "SJR 16", "A -A" → handle amendments
    cleaned = raw.upper()

    # CO bills often look like "HB21-1186" (House Bill, 2021, #1186)
    # Goal: Remove the "21-" part to get "HB1186"
    if state == 'CO':
        # Regex explanation:
        # ^([A-Z]+)   -> Capture prefix (HB, SB)
        # (\d{2})?    -> Optionally capture 2 digits (the year), e.g. "21"
        # -?          -> Optionally match a dash
        # (\d+)       -> Capture the actual bill number
        co_match = re.match(r'^([A-Z]+)(\d{2})?-?(\d+)$', cleaned)

        if co_match:
            prefix = co_match.group(1)
            # group(2) is the year (skip it)
            number = co_match.group(3)
            return f"{prefix}{number}"

    # Remove dots: H.B. → HB, S.J.R. → SJR, H.C.R. → HCR
    cleaned = re.sub(r'\.', '', cleaned)

    # Remove extra spaces and dashes between prefix parts
    # "H J R E 1" → "HJRE 1"
    # But preserve the space between prefix and number

    # Split into prefix (letters) and number
    match = re.match(r'^([A-Z\s]+?)\s*(\d+)\s*(.*)$', cleaned.strip())
    if not match:
        # Try handling amendment suffixes like "A -A", "S A"
        match = re.match(r'^([A-Z]+)\s*[-]?\s*([A-Z]?)$', cleaned.strip())
        if match:
            # This might be "A -A" meaning Assembly bill amendment
            # Try extracting differently
            num_match = re.search(r'(\d+)', raw)
            if num_match:
                prefix_part = re.sub(r'[\d\s.\-]', '', cleaned).strip()
                return normalize_prefix(prefix_part, state) + num_match.group(1)
        return raw.replace(' ', '').replace('.', '')

    prefix_raw = match.group(1).replace(' ', '').strip()
    number = match.group(2).strip()
    suffix = match.group(3).strip()  # e.g., amendment letters

    # --- Apply state-specific prefix mapping ---
    prefix = normalize_prefix(prefix_raw, state)

    # --- Handle leading zeros ---
    # Most states: no leading zeros (HB1, not HB001)
    # CT: uses leading zeros for 5-digit format (HB05934)
    # Some states might pad to 4 digits
    # LegiScan generally does NOT use leading zeros
    number = str(int(number))  # Remove any leading zeros

    # CT special case: LegiScan uses leading zeros for CT (e.g., HB05934)
    if state == 'CT':
        number = number.zfill(5)  # Pad to 5 digits

    result = f"{prefix}{number}"

    # Handle suffix (rare, mainly NY amendments like "A 691-A")
    if suffix and re.match(r'^[A-Z]$', suffix):
        result += suffix

    return result


def normalize_prefix(prefix_raw, state):
    """Map raw prefix to LegiScan format based on state."""
    # Check state-specific mapping first
    if state in STATE_PREFIX_MAP:
        mapping = STATE_PREFIX_MAP[state]
        if prefix_raw in mapping:
            return mapping[prefix_raw]

    # Default mappings for most states
    default_map = {
        'HB': 'HB', 'SB': 'SB',
        'AB': 'AB',
        'HF': 'HF', 'SF': 'SF',
        'HJR': 'HJR', 'SJR': 'SJR',
        'HCR': 'HCR', 'SCR': 'SCR',
        'HR': 'HR', 'SR': 'SR',
        'HJRE': 'HJRE',
        'HSB': 'HSB', 'SSB': 'SSB',
        'LB': 'LB', 'LD': 'LD',
        'HP': 'HP',
        'H': 'HB', 'S': 'SB',  # Default: H→HB, S→SB for most states
        'A': 'A',
        'PS': 'PS', 'PC': 'PC',
        'HD': 'HD',
    }

    # For states that use just H/S in LegiScan, override defaults
    h_s_states = {'FL', 'ID', 'KS', 'MA', 'NC', 'NJ', 'NY', 'RI', 'SC', 'VT'}
    if state in h_s_states:
        if prefix_raw in ('HB', 'H'):
            return STATE_PREFIX_MAP[state].get('HB', STATE_PREFIX_MAP[state].get('H', prefix_raw))
        if prefix_raw in ('SB', 'S'):
            return STATE_PREFIX_MAP[state].get('SB', STATE_PREFIX_MAP[state].get('S', prefix_raw))
        if prefix_raw in ('AB', 'A'):
            return STATE_PREFIX_MAP[state].get('AB', STATE_PREFIX_MAP[state].get('A', prefix_raw))

    return default_map.get(prefix_raw, prefix_raw)


# --- Apply normalization ---
aclu['bill_number'] = aclu.apply(
    lambda row: normalize_bill_number(row['bill_name'], row['state']), axis=1
)

# --- Verification ---
print(f"\n{'='*60}")
print("BILL NUMBER NORMALIZATION RESULTS")
print(f"{'='*60}")

# Show before/after for tricky states
tricky_states = ['CA', 'CO', 'CT', 'FL', 'IA', 'ID', 'KS', 'MA', 'ME',
                 'MN', 'NC', 'NE', 'NJ', 'NY', 'RI', 'SC', 'VT', 'WY']
for state in sorted(tricky_states):
    state_bills = aclu[aclu['state'] == state][[
        'bill_name', 'bill_number']].drop_duplicates()
    if len(state_bills) > 0:
        print(f"\n  {state}:")
        for _, row in state_bills.head(5).iterrows():
            print(f"    '{row['bill_name']}' → '{row['bill_number']}'")


# --- Deduplication ---
# ACLU tracks bills under multiple issues, creating duplicate rows
# Deduplicate on (year, state, bill_number) keeping label
print(f"\nBefore dedup: {len(aclu)} rows")

# If a bill appears as both harmful and supportive (shouldn't happen, but safety check),
# prioritize the label from the most specific issue tag
aclu_deduped = aclu.drop_duplicates(
    subset=['year', 'state', 'bill_number', 'label'])

# If same bill has conflicting labels, keep as-is (investigate manually)
dupes = aclu_deduped.groupby(
    ['year', 'state', 'bill_number']).filter(lambda x: len(x) > 1)
if len(dupes) > 0:
    print(f"\nWARNING: {len(dupes)} bills have conflicting labels:")
    print(dupes[['year', 'state', 'bill_number',
          'label', 'issues']].to_string(index=False))

# Final dedup: one row per (year, state, bill_number)
aclu_final = aclu_deduped.drop_duplicates(
    subset=['year', 'state', 'bill_number'], keep='first')
print(f"After dedup: {len(aclu_final)} rows")

print(f"\nFinal label distribution:\n{aclu_final['label'].value_counts()}")


# ============================================================
# SAVE
# ============================================================
aclu_final.to_csv('aclu_bills_cleaned.csv', index=False)
print(f"\nSaved → aclu_bills_cleaned.csv")


Total ACLU rows: 2515
Original label distribution:
label
harmful    2515
Name: count, dtype: int64

Corrected label distribution:
label
harmful       2409
supportive     106
Name: count, dtype: int64

--- Supportive Bills Sample ---
 year state bill_name                                                       issues
 2021    AZ   SB 1163 LGBTQ Equality Bills | Allowing updated gender markers on ID
 2021    AZ   HB 2652                      LGBTQ Equality Bills | Other good bills
 2021    KY    HB 130         LGBTQ Equality Bills | Nondiscrimination protections
 2021    KY    HB 116         LGBTQ Equality Bills | Nondiscrimination protections
 2021    MO   HB 1760         LGBTQ Equality Bills | Nondiscrimination protections
 2021    MO   HB 1737         LGBTQ Equality Bills | Nondiscrimination protections
 2021    MS   SB 2089         LGBTQ Equality Bills | Nondiscrimination protections
 2021    MS    HB 806         LGBTQ Equality Bills | Nondiscrimination protections
 2021    MS    HB 80

# Plural Data: Load, Normalize, and Merge with ACLU

Combines `anti_lgbtq_bills.csv` and `pro_lgbtq_bills.csv` from the Plural folder,
normalizes state names and bill numbers using the same logic as ACLU above,
then merges with the cleaned ACLU data to create a unified dataset.

In [ ]:
# ============================================================
# STEP 1: Load and combine Plural CSVs
# ============================================================
anti = pd.read_csv('../plural/anti_lgbtq_bills.csv')
anti['label'] = 'harmful'

pro = pd.read_csv('../plural/pro_lgbtq_bills.csv')
pro['label'] = 'supportive'

plural = pd.concat([anti, pro], ignore_index=True)
plural['source'] = 'plural'

print(f"Anti bills:  {len(anti)}")
print(f"Pro bills:   {len(pro)}")
print(f"Combined:    {len(plural)}")
print(f"\nColumns: {list(plural.columns)}")
print(f"\nLabel split:\n{plural['label'].value_counts()}")
print(f"\nStatus values:\n{plural['status'].value_counts()}")
plural.head(3)

Anti bills:  420
Pro bills:   180
Combined:    600

Columns: ['status', 'state', 'bill_number', 'description', 'sponsors', 'committee', 'session', 'latest_action', 'bill_id', 'label', 'source']

Label split:
label
harmful       420
supportive    180
Name: count, dtype: int64

Status values:
status
INTRODUCED               470
SIGNED BY GOVERNOR        34
BECAME LAW                33
PASSED UPPER              20
PASSED LOWER              17
PASSED                    12
VETOED                    10
REFERRED TO COMMITTEE      4
Name: count, dtype: int64


,status,state,bill_number,description,sponsors,committee,session,latest_action,bill_id,label,source
0,INTRODUCED,California,AB 600,Pupil instruction: transgender concepts: opt out.,Leticia Castillo (R),(NO COMMITTEE),2025-2026 REGULAR SESSION\nLATEST ACTION: FEBR...,"FEBRUARY 2, 2026",state-ca-20252026-ab600,harmful,plural
1,INTRODUCED,California,AB 281,Comprehensive sexual health education and huma...,James Gallagher (R),EDUCATION,2025-2026 REGULAR SESSION\nLATEST ACTION: FEBR...,"FEBRUARY 2, 2026",state-ca-20252026-ab281,harmful,plural
2,PASSED UPPER,Tennessee,SB 1424,"Obscenity and Pornography - As introduced, exp...",Joey Hensley (R),JUDICIARY,114TH REGULAR SESSION (2025-2026)\nLATEST ACTI...,"JANUARY 27, 2026",state-tn-114-sb1424,harmful,plural


In [ ]:
# ============================================================
# STEP 2: Normalize Plural states (full name → 2-letter abbrev)
# ============================================================
# Reuse STATE_TO_ABBR and normalize_state from cell 5
plural['state'] = plural['state'].apply(normalize_state)

unmapped = plural['state'].isna().sum()
if unmapped > 0:
    # Show what didn't map before dropping
    bad = plural[plural['state'].isna()]['state'].unique()
    print(f"WARNING: {unmapped} rows with unmapped states: {bad}")
    plural = plural.dropna(subset=['state'])

print(f"Plural states ({plural['state'].nunique()}): {sorted(plural['state'].unique())}")
print(f"Rows after state normalization: {len(plural)}")

Plural states (50): ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']
Rows after state normalization: 600


In [ ]:
# ============================================================
# STEP 3: Extract year from Plural session field
# ============================================================
# Session formats:
#   "2025-2026 REGULAR SESSION\nLATEST ACTION: ..."
#   "114TH REGULAR SESSION (2025-2026)\nLATEST ACTION: ..."
#   "104TH REGULAR SESSION\nLATEST ACTION: JANUARY 21, 2025"
#
# Strategy: extract the first 4-digit year, fallback to latest_action date

def extract_plural_year(row):
    """Extract year from session string, fallback to latest_action."""
    session = str(row.get('session', ''))
    # Try to find a year range like 2025-2026 → take 2025
    match = re.search(r'(20\d{2})', session)
    if match:
        return int(match.group(1))

    # Fallback: parse from latest_action date
    action = str(row.get('latest_action', ''))
    match = re.search(r'(20\d{2})', action)
    if match:
        return int(match.group(1))

    return pd.NA


plural['year'] = plural.apply(extract_plural_year, axis=1).astype('Int64')

print(f"Year coverage: {plural['year'].notna().sum()} / {len(plural)}")
print(f"Year range: {plural['year'].min()} – {plural['year'].max()}")
print(f"\nYear distribution:\n{plural['year'].value_counts().sort_index()}")

Year coverage: 600 / 600
Year range: 2024 – 2026

Year distribution:
year
2024      4
2025    594
2026      2
Name: count, dtype: Int64


In [ ]:
# ============================================================
# STEP 4: Normalize Plural bill numbers (same logic as ACLU)
# ============================================================
# Plural bill_number format: "AB 600", "SB 1424", "S 1505", "HB 1604"
# Need to normalize using the same normalize_bill_number() from cell 9

# The Plural data uses 'bill_number' column (with space), treat as bill_name
plural['bill_name'] = plural['bill_number']  # keep original
plural['bill_number'] = plural.apply(
    lambda row: normalize_bill_number(row['bill_name'], row['state']), axis=1
)

# Show normalization results for key states
print("Plural bill number normalization samples:")
for state in ['CA', 'MA', 'NY', 'TX', 'FL', 'ID', 'CT', 'CO']:
    subset = plural[plural['state'] == state][['bill_name', 'bill_number']].drop_duplicates()
    if len(subset) > 0:
        print(f"\n  {state}:")
        for _, row in subset.head(4).iterrows():
            print(f"    '{row['bill_name']}' → '{row['bill_number']}'")

print(f"\nTotal Plural bills after normalization: {len(plural)}")

Plural bill number normalization samples:

  CA:
    'AB 600' → 'AB600'
    'AB 281' → 'AB281'
    'AB 89' → 'AB89'
    'SB 59' → 'SB59'

  MA:
    'H 550' → 'H550'
    'H 551' → 'H551'
    'H 737' → 'H737'
    'HD 4542' → 'HD4542'

  NY:
    'S 7489' → 'S7489'
    'S 3591' → 'S3591'
    'A 8053' → 'A8053'
    'A 3946' → 'A3946'

  TX:
    'SB 13' → 'SB13'
    'HB 581' → 'HB581'
    'HB 229' → 'HB229'
    'HB 1106' → 'HB1106'

  FL:
    'SB 440' → 'SB440'
    'SB 1288' → 'SB1288'
    'HB 1505' → 'HB1505'
    'HB 1495' → 'HB1495'

  ID:
    'H 352' → 'H352'
    'H 59' → 'H59'
    'H 351' → 'H351'
    'H 292' → 'H292'

  CT:
    'HB 7014' → 'HB07014'
    'SB 1380' → 'SB01380'
    'HB 6913' → 'HB06913'
    'HB 7135' → 'HB07135'

  CO:
    'SB 25-201' → 'SB25'
    'HB 25-1253' → 'HB25'
    'HB 25-1255' → 'HB25'
    'HB 25-1254' → 'HB25'

Total Plural bills after normalization: 600


In [ ]:
# ============================================================
# STEP 5: Normalize Plural status to match ACLU format
# ============================================================
# Plural statuses: INTRODUCED, PASSED UPPER, PASSED LOWER, SIGNED,
#                  PASSED, FAILED, VETOED, etc.
# Map to ACLU-style: Introduced, Passed, Failed, Signed, Vetoed

STATUS_MAP = {
    'INTRODUCED':           'Introduced',
    'REFERRED TO COMMITTEE':'Introduced',
    'ENGROSSED':            'Introduced',
    'PASSED UPPER':         'Passed',
    'PASSED LOWER':         'Passed',
    'PASSED':               'Passed',
    'ENROLLED':             'Passed',
    'SIGNED':               'Signed',
    'SIGNED BY GOVERNOR':   'Signed',
    'BECAME LAW':           'Signed',
    'VETOED':               'Vetoed',
    'FAILED':               'Failed',
    'DEAD':                 'Failed',
}

plural['status_normalized'] = plural['status'].str.strip().str.upper().map(STATUS_MAP)

# Flag any unmapped statuses before filling
unmapped_statuses = plural[plural['status_normalized'].isna()]['status'].unique()
if len(unmapped_statuses) > 0:
    print(f"WARNING: unmapped statuses (defaulting to 'Introduced'): {list(unmapped_statuses)}")
plural['status_normalized'] = plural['status_normalized'].fillna('Introduced')

# Parse latest_action as status_date
plural['status_date'] = pd.to_datetime(
    plural['latest_action'], format='mixed', dayfirst=False, errors='coerce'
)

print(f"Plural status mapping:")
print(plural.groupby(['status', 'status_normalized']).size().to_string())
print(f"\nStatus date coverage: {plural['status_date'].notna().sum()} / {len(plural)}")

Plural status mapping:
status                 status_normalized
BECAME LAW             Signed                33
INTRODUCED             Introduced           470
PASSED                 Passed                12
PASSED LOWER           Passed                17
PASSED UPPER           Passed                20
REFERRED TO COMMITTEE  Introduced             4
SIGNED BY GOVERNOR     Signed                34
VETOED                 Vetoed                10

Status date coverage: 599 / 600


In [ ]:
# ============================================================
# STEP 6: Build Plural dataframe with ACLU-compatible columns
# ============================================================
plural_clean = plural[[
    'year', 'state', 'bill_name', 'bill_number', 'label', 'source',
    'description', 'sponsors', 'status_normalized', 'status_date'
]].rename(columns={
    'status_normalized': 'status',
}).copy()

# Add columns that exist in ACLU but not Plural (fill with NA)
plural_clean['issues'] = pd.NA
plural_clean['status_detail'] = plural['status']  # keep original Plural status as detail

print(f"Plural clean shape: {plural_clean.shape}")
print(f"Columns: {list(plural_clean.columns)}")
plural_clean.head(3)

Plural clean shape: (600, 12)
Columns: ['year', 'state', 'bill_name', 'bill_number', 'label', 'source', 'description', 'sponsors', 'status', 'status_date', 'issues', 'status_detail']


,year,state,bill_name,bill_number,label,source,description,sponsors,status,status_date,issues,status_detail
0,2025,CA,AB 600,AB600,harmful,plural,Pupil instruction: transgender concepts: opt out.,Leticia Castillo (R),Introduced,2026-02-02,<NA>,INTRODUCED
1,2025,CA,AB 281,AB281,harmful,plural,Comprehensive sexual health education and huma...,James Gallagher (R),Introduced,2026-02-02,<NA>,INTRODUCED
2,2025,TN,SB 1424,SB1424,harmful,plural,"Obscenity and Pornography - As introduced, exp...",Joey Hensley (R),Passed,2026-01-27,<NA>,PASSED UPPER


In [ ]:
# ============================================================
# STEP 7: Merge ACLU cleaned + Plural cleaned
# ============================================================
# Load the ACLU cleaned data (from cell 9 output)
aclu_clean = pd.read_csv('aclu_bills_cleaned.csv')
aclu_clean['source'] = 'aclu'

# Add columns that Plural has but ACLU doesn't
aclu_clean['description'] = pd.NA
aclu_clean['sponsors'] = pd.NA

print(f"ACLU clean:   {len(aclu_clean)} rows")
print(f"Plural clean: {len(plural_clean)} rows")

# Align columns for concat
shared_cols = ['year', 'state', 'bill_name', 'bill_number', 'issues',
               'label', 'source', 'status', 'status_detail', 'status_date',
               'description', 'sponsors']

combined = pd.concat([
    aclu_clean.reindex(columns=shared_cols),
    plural_clean.reindex(columns=shared_cols),
], ignore_index=True)

print(f"\nCombined (before dedup): {len(combined)} rows")
print(f"Source split:\n{combined['source'].value_counts()}")
print(f"\nLabel split:\n{combined['label'].value_counts()}")

ACLU clean:   2496 rows
Plural clean: 600 rows

Combined (before dedup): 3096 rows
Source split:
source
aclu      2496
plural     600
Name: count, dtype: int64

Label split:
label
harmful       2811
supportive     285
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 8: Deduplicate — same bill from both sources
# ============================================================
# If a bill appears in both ACLU and Plural, prefer ACLU (richer metadata)
# Match on (state, bill_number) — year may differ slightly between sources

# First, find overlaps
combined['_dedup_key'] = combined['state'] + '_' + combined['bill_number'].astype(str)

overlap = (combined.groupby('_dedup_key')['source']
           .apply(lambda x: set(x))
           .reset_index(name='sources'))
both_sources = overlap[overlap['sources'] == {'aclu', 'plural'}]
print(f"Bills appearing in BOTH sources: {len(both_sources)}")

# Show some overlap examples
if len(both_sources) > 0:
    sample_keys = both_sources['_dedup_key'].head(10).tolist()
    print(f"\nOverlap samples:")
    for key in sample_keys[:5]:
        rows = combined[combined['_dedup_key'] == key][
            ['source', 'state', 'bill_number', 'year', 'label', 'status']
        ]
        print(rows.to_string(index=False))
        print()

# Deduplicate: sort so ACLU comes first (preferred), then drop duplicates
combined = combined.sort_values('source', ascending=True)  # 'aclu' < 'plural'
before = len(combined)
combined = combined.drop_duplicates(subset=['state', 'bill_number'], keep='first')
combined = combined.drop(columns=['_dedup_key'])

print(f"\nDeduplication: {before} → {len(combined)} rows ({before - len(combined)} removed)")
print(f"\nFinal source split:\n{combined['source'].value_counts()}")
print(f"\nFinal label split:\n{combined['label'].value_counts()}")

Bills appearing in BOTH sources: 256

Overlap samples:
source state bill_number  year   label     status
  aclu    AK        HB40  2025 harmful  Advancing
  aclu    AK        HB40  2026 harmful Introduced
plural    AK        HB40  2025 harmful Introduced

source state bill_number  year   label     status
  aclu    AL       HB107  2025 harmful   Defeated
plural    AL       HB107  2025 harmful Introduced

source state bill_number  year   label     status
  aclu    AL        HB23  2026 harmful  Advancing
plural    AL        HB23  2026 harmful Introduced

source state bill_number  year   label   status
  aclu    AL       HB244  2025 harmful Defeated
plural    AL       HB244  2025 harmful   Passed

source state bill_number  year   label     status
  aclu    AL       HB246  2025 harmful   Defeated
plural    AL       HB246  2025 harmful Introduced


Deduplication: 3096 → 2312 rows (784 removed)

Final source split:
source
aclu      1977
plural     335
Name: count, dtype: int64

Final label sp

In [ ]:
# ============================================================
# STEP 9: Final sanity checks and save
# ============================================================
print("=" * 60)
print("FINAL MERGED DATASET")
print("=" * 60)

print(f"\nShape: {combined.shape}")
print(f"Columns: {list(combined.columns)}")

print(f"\nNull counts:")
print(combined.isnull().sum())

print(f"\nLabel × Source:")
print(combined.groupby(['source', 'label']).size().unstack(fill_value=0))

print(f"\nYear distribution:")
print(combined['year'].value_counts().sort_index())

print(f"\nStates: {combined['state'].nunique()}")
print(f"Unique bills: {combined[['state', 'bill_number']].drop_duplicates().shape[0]}")

# Save
combined.to_csv('all_lgbtq_bills_merged.csv', index=False)
print(f"\nSaved → all_lgbtq_bills_merged.csv ({len(combined)} rows)")

FINAL MERGED DATASET

Shape: (2312, 12)
Columns: ['year', 'state', 'bill_name', 'bill_number', 'issues', 'label', 'source', 'status', 'status_detail', 'status_date', 'description', 'sponsors']

Null counts:
year                0
state               0
bill_name           0
bill_number         0
issues            335
label               0
source              0
status             12
status_detail     241
status_date        17
description      1977
sponsors         2005
dtype: int64

Label × Source:
label   harmful  supportive
source                     
aclu       1887          90
plural      161         174

Year distribution:
year
2021    154
2022    221
2023    394
2024    444
2025    944
2026    155
Name: count, dtype: Int64

States: 51
Unique bills: 2312

Saved → all_lgbtq_bills_merged.csv (2312 rows)
